In [36]:
import requests
import json
import os
import urllib3
import boto3
import io
import gzip
import logging
import pandas as pd
from datetime import datetime
from pandas.tseries.holiday import USFederalHolidayCalendar

In [37]:
s3_client = boto3.client('s3')
today_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
bucket_name = "uncc_traffic_data"

In [38]:
s3_bucket = "s3-3601-final-project"
s3_key = f"tomtom/traffic/{today_str}speeds.json.gz"

In [39]:
api_key = 'KFAr7dIvLb4Fw2ntylN7NFGLqRIJXYxc'
#timestamp = datetime.utcnow()


url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"

data_list = []

latitude = [35.307778, 35.313611, 35.311361, 35.309306, 35.306917, 35.293111, 35.294889, 35.299639, 35.287972, 35.301861, 35.311861] # fill in with the found values
longitude = [-80.724583, -80.701944, -80.711222, -80.718056, -80.722639, -80.749361, -80.742111, -80.74575, -80.759472, -80.752333, -80.744]

for i in range(len(latitude)):

        params = {
            'key': api_key,
            'point': f"{latitude[i]},{longitude[i]}",
            'unit': "MPH",
            "openLr": "false"
        }

        response = requests.get(url, params)

        if response.status_code==200:
            data = response.json()
            flow_data = data.get("flowSegmentData", {})

            data_we_want = {
                "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
                "latitude": latitude[i],
                "longitude": longitude[i],
                "current_speed": flow_data.get("currentSpeed"),
                "free_flow_speed": flow_data.get("freeFlowSpeed"),
                "current_delay": flow_data.get("currentDelay"),
                "confidence": flow_data.get("confidence") # for flags of when we shoudl somehow retrain?
            }

            data_list.append(data_we_want)
        else:
            print(f"Error in retrieving data: {response.status_code}")
            
            #making data_list into a DataFrame
df = pd.DataFrame(data_list) 
print(df.head())
if not df.empty and "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%Y-%m-%d_%H-%M-%S"
    )
    df["day_of_week"] = df["timestamp"].dt.day_name()

             timestamp   latitude  longitude  current_speed  free_flow_speed  \
0  2026-09-26_02-43-13  35.307778 -80.724583             27               27   
1  2026-09-26_02-43-14  35.313611 -80.701944             40               40   
2  2026-09-26_02-43-15  35.311361 -80.711222             31               31   
3  2026-09-26_02-43-17  35.309306 -80.718056             31               31   
4  2026-09-26_02-43-18  35.306917 -80.722639             31               31   

  current_delay  confidence  
0          None           1  
1          None           1  
2          None           1  
3          None           1  
4          None           1  


In [35]:
df.dtypes

timestamp          datetime64[ns]
latitude                  float64
longitude                 float64
current_speed               int64
free_flow_speed             int64
current_delay              object
confidence                  int64
day_of_week                 int32
hour                        int32
month                       int32
dtype: object

In [51]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
# Create time features
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month

df.head(11)

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,speed_ratio,holiday,hour,month
0,2026-09-26 02:43:13,35.307778,-80.724583,27,27,None,1,5,1.000000,Not a federal holiday,2,9
1,2026-09-26 02:43:14,35.313611,-80.701944,40,40,None,1,5,1.000000,Not a federal holiday,2,9
2,2026-09-26 02:43:15,35.311361,-80.711222,31,31,None,1,5,1.000000,Not a federal holiday,2,9
3,2026-09-26 02:43:17,35.309306,-80.718056,31,31,None,1,5,1.000000,Not a federal holiday,2,9
4,2026-09-26 02:43:18,35.306917,-80.722639,31,31,None,1,5,1.000000,Not a federal holiday,2,9
5,2026-09-26 02:43:19,35.293111,-80.749361,29,29,None,1,5,1.000000,Not a federal holiday,2,9
6,2026-09-26 02:43:20,35.294889,-80.742111,55,55,None,1,5,1.000000,Not a federal holiday,2,9
7,2026-09-26 02:43:22,35.299639,-80.745750,22,28,None,1,5,0.785714,Not a federal holiday,2,9
8,2026-09-26 02:43:23,35.287972,-80.759472,25,25,None,1,5,1.000000,Not a federal holiday,2,9
9,2026-09-26 02:43:24,35.301861,-80.752333,27,27,None,1,5,1.000000,Not a federal holiday,2,9


In [40]:
#converting into a csv to save into our shared s3 bucket
df.to_csv("TomTom_traffic.csv", index=False)


s3_client.upload_file(
    "TomTom_traffic.csv",
    "s3-3601-final-project",
    "tomtom/traffic/speeds.json.gz"
)

In [41]:
compressed = io.BytesIO()
with gzip.GzipFile(fileobj=compressed, mode="wb") as gz:
    gz.write(json.dumps(data_list).encode("utf-8"))
                                    
s3_client.put_object(
    Bucket=s3_bucket,
    Key=s3_key,
    Body=compressed.getvalue(),
    ContentType="application/json",
    ContentEncoding="gzip",
)

{'ResponseMetadata': {'RequestId': 'DFJY9WD4343TXMWX',
  'HostId': 'Cmm6sWXXi2rdx3sGA/KI0vmJUdKe/3f/RY29Q/BtW2qKRCHWucoCUeL/lCzD7L/FkznLDaxv8JY=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'Cmm6sWXXi2rdx3sGA/KI0vmJUdKe/3f/RY29Q/BtW2qKRCHWucoCUeL/lCzD7L/FkznLDaxv8JY=',
   'x-amz-request-id': 'DFJY9WD4343TXMWX',
   'date': 'Sat, 26 Sep 2026 06:43:40 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"bd50fd3429c4bcfd1abed5495b3b6d11"',
   'x-amz-checksum-crc32': 'b3JRzg==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"bd50fd3429c4bcfd1abed5495b3b6d11"',
 'ChecksumCRC32': 'b3JRzg==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [42]:
df["speed_ratio"] = (df["current_speed"] / df["free_flow_speed"])

In [50]:
calendar = USFederalHolidayCalendar()
    #To use after getting traffic data from holidays
holiday_names = calendar.holidays(
        start=df["timestamp"].min().normalize(),
        end=df["timestamp"].max().normalize(),
        return_name=True,
    )

df["holiday"] = (
        #map timestamp to the correct holiday name
        df["timestamp"].dt.normalize()
        .map(holiday_names)
        .fillna("Not a federal holiday")
    )
if not df.empty:
    holiday_names = calendar.holidays(
        start=df["timestamp"].min().normalize(),
        end=df["timestamp"].max().normalize(),
        return_name=True,
    )

    df["holiday"] = (
        df["timestamp"].dt.normalize()
        .map(holiday_names)
        .fillna("Not a federal holiday")
    )
else:
    print("No traffic rows available.")